# SEOCHO Quickstart + Core Features

This notebook is the recommended first entry route for new users.

You will walk through the core product contract in one place:

- ontology-first setup
- indexing design YAML inspection
- agent design YAML inspection
- local indexing into embedded LadybugDB or optional Neo4j/DozerDB over Bolt
- ontology-aware querying with observability metadata
- side-by-side runs across four LLM providers

Important:

- Open this notebook from the repository's `examples/` directory.
- Fill your `.env` first if you want to run the model comparison section.
- The four-model section uses real API calls and costs real tokens.


## Before You Start

Recommended install for this notebook:

```bash
uv pip install "seocho[local]" python-dotenv jupyter
```

Then prepare your environment file:

```bash
cp ../.env.example ../.env
```

Supported provider env vars used below:

- `OPENAI_API_KEY`
- `DEEPSEEK_API_KEY`
- `MOONSHOT_API_KEY`
- `XAI_API_KEY` or `GROK_API_KEY`

Optional graph env vars for the production-like local path:

- `NEO4J_URI`
- `NEO4J_USER`
- `NEO4J_PASSWORD`

If `NEO4J_URI` is unset, the notebook uses embedded LadybugDB by default. If it is set, the local client switches to Neo4j/DozerDB over Bolt.


In [ ]:
# %pip install "seocho[local]" python-dotenv jupyter

from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

from seocho import (
    Seocho,
    list_provider_specs,
    load_agent_design_spec,
    load_indexing_design_spec,
)

ROOT = Path.cwd()
assert (ROOT / "agent_designs").exists(), "Run this notebook from the examples/ directory."

ENV_PATH = ROOT.parent / ".env"
load_dotenv(ENV_PATH)
sys.path.insert(0, str((ROOT / "finance-compliance").resolve()))

from ontology import build_ontology

RUNS_DIR = ROOT / ".notebook_runs"
RUNS_DIR.mkdir(exist_ok=True)

PRIMARY_MODELS = [
    ("openai", "gpt-4o-mini"),
    ("deepseek", "deepseek-chat"),
    ("kimi", "kimi-k2.5"),
    ("grok", "grok-4.20-reasoning"),
]


In [ ]:
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
USE_NEO4J = bool(NEO4J_URI)
GRAPH_BACKEND = "neo4j_bolt" if USE_NEO4J else "ladybug_embedded"
GRAPH_TARGET_HINT = NEO4J_URI or str(RUNS_DIR / "<suffix>-<provider>-<model>.lbug")

env_summary = {
    "env_path": str(ENV_PATH),
    "env_exists": ENV_PATH.exists(),
    "graph_backend": GRAPH_BACKEND,
    "graph_target_hint": GRAPH_TARGET_HINT,
    "neo4j_user": NEO4J_USER if USE_NEO4J else None,
    "neo4j_password_configured": bool(NEO4J_PASSWORD) if USE_NEO4J else False,
}
pprint(env_summary)
print()

provider_specs = list_provider_specs()
provider_rows = []

for provider, model in PRIMARY_MODELS:
    spec = provider_specs[provider]
    aliases = [spec.api_key_env, *spec.api_key_env_aliases]
    configured = any(bool(os.getenv(name)) for name in aliases)
    provider_rows.append(
        {
            "provider": provider,
            "model": model,
            "api_key_envs": aliases,
            "configured": configured,
            "base_url": spec.base_url or "OpenAI default",
        }
    )

pprint(provider_rows)

default_row = next((row for row in provider_rows if row["configured"]), provider_rows[0])
DEFAULT_PROVIDER = default_row["provider"]
DEFAULT_MODEL = default_row["model"]

print(f"Default notebook provider: {DEFAULT_PROVIDER}/{DEFAULT_MODEL}")
print(f"Artifacts and local graph files will be written under: {RUNS_DIR}")


## 1. Define One Ontology Contract

SEOCHO starts from ontology, not prompt fragments.

This notebook uses the small finance-compliance example because it is realistic enough to query, but still small enough to understand quickly.


In [ ]:
ontology = build_ontology()
schema_path = RUNS_DIR / "finance_compliance_schema.jsonld"
ontology.to_jsonld(schema_path)

print(f"Ontology name: {ontology.name}")
print(f"Saved JSON-LD: {schema_path}")
print("Node types:", sorted(ontology.nodes.keys()))
print("Relationship types:", sorted(ontology.relationships.keys()))


In [ ]:
extraction_context = ontology.to_extraction_context()
query_context = ontology.to_query_context()

print("Extraction entity types:")
print(extraction_context["entity_types"])
print()
print("Query graph schema:")
print(query_context["graph_schema"])
print()
print("Query hints:")
print(query_context["query_hints"])


## 2. Inspect Indexing Design Specs

Indexing design specs make graph-model choices explicit. They are the right place to state whether you want LPG, RDF, or hybrid ingestion behavior.

For the runnable local path in this notebook, we use the LPG + Ladybug design because it is the lightest first-run setup.


In [ ]:
indexing_design_paths = sorted((ROOT / "indexing_designs").glob("*.yaml"))
indexing_design_rows = []

for path in indexing_design_paths:
    spec = load_indexing_design_spec(path)
    compiled = spec.client_kwargs(ontology=ontology)
    indexing_design_rows.append(
        {
            "file": path.name,
            "graph_model": spec.graph_model,
            "storage_target": spec.storage_target,
            "ontology_profile": compiled["ontology_profile"],
            "require_workspace_id": spec.requires_workspace_id(),
        }
    )

pprint(indexing_design_rows)

ACTIVE_INDEXING_DESIGN = ROOT / "indexing_designs" / "lpg_finance_provenance.yaml"
active_indexing_spec = load_indexing_design_spec(ACTIVE_INDEXING_DESIGN)
print(f"Active indexing design: {ACTIVE_INDEXING_DESIGN.name}")


## 3. Inspect Agent Design Specs

Agent design specs keep pattern choices reviewable in YAML. The key guardrail is that every design must bind to an ontology.

Below we inspect the three included examples and then choose a reflection-style design for the runnable query pass.


In [ ]:
agent_design_paths = sorted((ROOT / "agent_designs").glob("*.yaml"))
agent_rows = []

for path in agent_design_paths:
    spec = load_agent_design_spec(path)
    config = spec.to_agent_config()
    agent_rows.append(
        {
            "file": path.name,
            "pattern": spec.pattern,
            "ontology_profile": spec.client_kwargs()["ontology_profile"],
            "execution_mode": config.execution_mode,
            "reasoning_mode": config.reasoning_mode,
            "repair_budget": config.repair_budget,
        }
    )

pprint(agent_rows)

ACTIVE_AGENT_DESIGN = ROOT / "agent_designs" / "reflection_chain_finance.yaml"
print(f"Active agent design: {ACTIVE_AGENT_DESIGN.name}")


## 4. Build a Local Client and Index Sample Documents

This section uses:

- the finance-compliance ontology
- the LPG provenance indexing design
- the first configured provider from your environment
- embedded LadybugDB by default, or Neo4j/DozerDB over Bolt when `NEO4J_URI` is set

We intentionally keep the first run small and cheap: three documents are enough to see the full indexing and querying path.


In [ ]:
SAMPLE_DOCS = sorted((ROOT / "finance-compliance" / "sample_docs").glob("*.txt"))
DATABASE = "quickstart_demo"

def make_local_client(provider=DEFAULT_PROVIDER, model=DEFAULT_MODEL, *, suffix="default", agent_design=None):
    graph_path = RUNS_DIR / f"{suffix}-{provider}-{model.replace('/', '-').replace('.', '-')}.lbug"
    if not USE_NEO4J and graph_path.exists():
        graph_path.unlink()

    kwargs = {
        "ontology": ontology,
        "llm": f"{provider}/{model}",
        "graph": NEO4J_URI if USE_NEO4J else str(graph_path),
        "workspace_id": f"notebook-{suffix}",
    }
    if USE_NEO4J:
        if not NEO4J_PASSWORD:
            raise RuntimeError("Set NEO4J_PASSWORD in .env before using the Neo4j notebook path.")
        kwargs["neo4j_user"] = NEO4J_USER
        kwargs["neo4j_password"] = NEO4J_PASSWORD

    if agent_design is not None:
        return Seocho.from_agent_design(agent_design, **kwargs)
    return Seocho.from_indexing_design(ACTIVE_INDEXING_DESIGN, **kwargs)

def index_sample_docs(client, *, database=DATABASE, limit=3):
    rows = []
    for path in SAMPLE_DOCS[:limit]:
        memory = client.add(
            path.read_text(),
            database=database,
            category="finance_compliance",
            metadata={"source_file": path.name},
        )
        rows.append(
            {
                "file": path.name,
                "status": memory.status,
                "nodes_created": memory.metadata.get("nodes_created", 0),
                "relationships_created": memory.metadata.get("relationships_created", 0),
                "fallback_used": memory.metadata.get("fallback_used", False),
            }
        )
    return rows

client = make_local_client(suffix="quickstart")
index_rows = index_sample_docs(client)
pprint(index_rows)


## 5. Query and Inspect Observability Metadata

Natural-language answers are useful, but onboarding is much stronger when the user can also see what SEOCHO thought it was doing.

Below we run a direct question, then a bounded repair-style question, and inspect the local query metadata that SEOCHO records.


In [ ]:
answer = client.ask(
    "Which regulations is Acme Financial Services subject to, and who enforces them?",
    database=DATABASE,
)
print(answer)


In [ ]:
reasoned_answer = client.ask(
    "Which control evidence mitigates the reported compliance incident?",
    database=DATABASE,
    reasoning_mode=True,
    repair_budget=2,
)
print(reasoned_answer)

metadata = client.last_query_metadata
observability_summary = {
    "ontology_profile": client.ontology_profile,
    "support_status": metadata.get("support_assessment", {}).get("status"),
    "missing_slots": metadata.get("evidence_bundle", {}).get("missing_slots"),
    "retrieval_ms": metadata.get("latency_breakdown_ms", {}).get("retrieval_ms"),
    "generation_ms": metadata.get("latency_breakdown_ms", {}).get("generation_ms"),
    "agent_pattern": metadata.get("agent_pattern", {}).get("pattern"),
    "token_usage": metadata.get("token_usage", {}),
}
pprint(observability_summary)


## 6. Apply an Agent Design Pattern

The indexing design controlled how facts were written.
The agent design controls how the client should behave when answering.

Here we switch to the `reflection_chain` YAML and let that design set the default reasoning posture for a local client.


In [ ]:
pattern_client = make_local_client(suffix="reflection", agent_design=ACTIVE_AGENT_DESIGN)
pattern_rows = index_sample_docs(pattern_client, database="pattern_demo", limit=3)
pprint(pattern_rows)

pattern_answer = pattern_client.ask(
    "Which policies govern trade surveillance at Acme Financial Services?",
    database="pattern_demo",
)
print(pattern_answer)

pattern_plan = (
    pattern_client.plan("Which regulations relate to the incident?")
    .react(max_steps=4, tool_budget=3)
    .with_repair_budget(2)
    .build()
)

pprint(pattern_plan.to_dict())
pprint(pattern_client.last_query_metadata.get("agent_pattern", {}))


## 7. Compare Four LLM Providers on the Same Workflow

This is the section most users ask for once the first local run works:

- same ontology
- same indexing design
- same documents
- same graph backend selected from `.env`
- same question
- only the provider/model changes

If a provider key is missing, the notebook records that and skips the run.


In [ ]:
COMPARISON_QUESTION = "Which regulations is Acme Financial Services subject to, and who enforces them?"

def run_model_matrix(limit_docs=2):
    results = []
    for provider, model in PRIMARY_MODELS:
        spec = provider_specs[provider]
        aliases = [spec.api_key_env, *spec.api_key_env_aliases]
        if not any(bool(os.getenv(name)) for name in aliases):
            results.append(
                {
                    "provider": provider,
                    "model": model,
                    "status": "skipped_missing_api_key",
                    "answer": "",
                }
            )
            continue

        trial_client = make_local_client(provider, model, suffix=f"matrix-{provider}")
        trial_database = f"matrix_{provider}"
        try:
            index_sample_docs(trial_client, database=trial_database, limit=limit_docs)
            answer = trial_client.ask(
                COMPARISON_QUESTION,
                database=trial_database,
                reasoning_mode=True,
                repair_budget=1,
            )
            metadata = trial_client.last_query_metadata
            results.append(
                {
                    "provider": provider,
                    "model": model,
                    "status": "ok",
                    "support_status": metadata.get("support_assessment", {}).get("status"),
                    "retrieval_ms": metadata.get("latency_breakdown_ms", {}).get("retrieval_ms"),
                    "generation_ms": metadata.get("latency_breakdown_ms", {}).get("generation_ms"),
                    "total_tokens_est": metadata.get("token_usage", {}).get("total_tokens_est"),
                    "agent_pattern": metadata.get("agent_pattern", {}).get("pattern"),
                    "answer": answer[:220],
                }
            )
        except Exception as exc:
            results.append(
                {
                    "provider": provider,
                    "model": model,
                    "status": f"error:{type(exc).__name__}",
                    "answer": str(exc),
                }
            )
        finally:
            trial_client.close()
    return results

model_results = run_model_matrix()
pprint(model_results)


## 8. What To Look At After The First Run

If the answer quality is weaker than expected, do not guess blindly.
Inspect these fields first:

- `support_status`
- `missing_slots`
- `retrieval_ms`
- `generation_ms`
- `agent_pattern`
- `token_usage`

That is the minimum useful loop for improving SEOCHO behavior without changing random prompts first.


In [ ]:
client.close()
pattern_client.close()
print("Closed local notebook clients.")
